In [1]:
import os
os.getcwd()

'C:\\Users\\ACER-PC\\Desktop\\WUIR-CLASS-recSys\\UIR\\Scripts'

In [2]:
import yaml

from Dataset import Dataset
import matchings

# Load config
with open("../config/run2.yaml", "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

dataset = Dataset(config)

In [3]:
import numpy as np
jobs = dataset.jobs
courses = dataset.courses
skills = dataset.skills
learners = dataset.learners

print(f"Jobs: {jobs},\n Skills: {skills},\n Courses: {courses},\n Learners: {learners}")

#print(np.sum(jobs > 0, axis=1))

Jobs: [[3 3 0 ... 0 0 0]
 [2 3 0 ... 0 0 0]
 [3 3 3 ... 0 0 0]
 ...
 [2 2 0 ... 0 0 0]
 [3 3 3 ... 0 0 0]
 [3 3 0 ... 0 1 0]],
 Skills: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45},
 Courses: [[[1 1 0 ... 0 0 0]
  [0 2 0 ... 0 0 0]]

 [[0 0 2 ... 0 0 0]
  [1 2 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [1 0 0 ... 0 0 0]]

 ...

 [[1 0 0 ... 0 0 0]
  [2 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 1 ... 0 0 0]]],
 Learners: [[2 2 0 ... 0 0 0]
 [0 2 0 ... 0 0 3]
 [1 0 0 ... 0 0 0]
 ...
 [1 1 0 ... 0 0 0]
 [2 2 0 ... 0 0 0]
 [1 2 0 ... 0 0 0]]


## Two course sequence

In [4]:
def maximize_jobs_2courses(learners, courses, dataset):
    total = []
    i = 0
    print(len(learners))
    for learner in learners:
        actual_jobs = dataset.get_nb_applicable_jobs(learner, 0.8)
        sequences = []
        max_job = actual_jobs
        for i, course1 in enumerate(courses):
            req = matchings.learner_course_required_matching(learner, course1)
            if req >= 0.8:
                learner_1course = np.maximum(learner, course1[1])
                new_max = dataset.get_nb_applicable_jobs(learner_1course, 0.8)
                for j, course2 in enumerate(courses):
                    req2 = matchings.learner_course_required_matching(learner_1course, course2)
                    if req2 >= 0.8:
                        learner_2course = np.maximum(learner_1course, course2[1])
                        new_max = dataset.get_nb_applicable_jobs(learner_2course, 0.8)
                        #print(f"New max job: {new_max}, previous max job: {actual_jobs}")
                        if new_max > max_job:
                            max_job = new_max
                            j_max = j
                            sequences = [i, j_max]
        if sequences != []:
            learner = np.maximum(learner, courses[sequences[0]][1])
            learner = np.maximum(learner, courses[sequences[1]][1])
        total.append(max_job)
        #print(f"For learner: {learner} we reached a number of max_job:{max_job}, with sequence: {sequences}")

    print(sum(total) / len(total))
    return sum(total) / len(total)
#print(dataset.get_avg_applicable_jobs(0.8))

In [5]:
print(maximize_jobs_2courses(learners, courses, dataset))

52
12.423076923076923
12.423076923076923


## Three Courses sequence

In [6]:
import numpy as np
from numba import njit, prange

# ---------- helpers nopython (coerenti con le tue funzioni) ----------

@njit(cache=True)
def matching_njit(level1: np.ndarray, level2: np.ndarray) -> float:
    acc = 0.0
    denom = 0
    for i in range(level1.size):
        req = level2[i]
        if req > 0.0:
            lv = level1[i]
            mn = lv if lv < req else req
            acc += (mn / req)
            denom += 1
    if denom == 0:
        return 0.0
    return acc / denom

@njit(cache=True)
def learner_course_required_matching_njit(learner: np.ndarray, course: np.ndarray) -> float:
    required_course = course[0]
    has_any = False
    for i in range(required_course.size):
        if required_course[i] != 0.0:
            has_any = True
            break
    if not has_any:
        return 1.0
    return matching_njit(learner, required_course)

@njit(cache=True)
def _nb_applicable_jobs_numba(learner: np.ndarray, jobs: np.ndarray, threshold: float) -> int:
    J, S = jobs.shape
    count = 0
    for j in range(J):
        denom = 0
        acc = 0.0
        for s in range(S):
            req = jobs[j, s]
            if req > 0:
                denom += 1
                lv = learner[s]
                acc += (lv if lv < req else req) / req
        if denom > 0:
            score = acc / denom
            if score >= threshold:
                count += 1
    return count

# ---------- kernel principale: miglior sequenza di 3 corsi ----------

@njit(parallel=True, cache=True)
def maximize_jobs_3courses_njit(
    learners: np.ndarray,        # (L, D)
    courses: np.ndarray,         # (C, 2, D) -> [required, provided]
    jobs: np.ndarray,            # (J, D)
    threshold: float = 0.8
):
    """
    Output:
      - max_jobs: (L,) int64
      - sequences: (L, 3) int64  -> indici [i,j,k] o -1 se nessuna sequenza migliora
      - final_learners: (L, D)
    """
    L, D = learners.shape
    C = courses.shape[0]

    max_jobs = np.empty(L, dtype=np.int64)
    sequences = np.full((L, 3), -1, dtype=np.int64)
    final_learners = np.empty((L, D), dtype=learners.dtype)

    for li in prange(L):
        base = learners[li]
        best_jobs = _nb_applicable_jobs_numba(base, jobs, threshold)
        bi = -1
        bj = -1
        bk = -1

        # buffer per evitare allocazioni ripetute
        tmp1 = np.empty(D, dtype=learners.dtype)
        tmp2 = np.empty(D, dtype=learners.dtype)
        tmp3 = np.empty(D, dtype=learners.dtype)

        for i in range(C):
            if learner_course_required_matching_njit(base, courses[i]) >= threshold:
                # tmp1 = max(base, provided1)
                for d in range(D):
                    a = base[d]
                    b = courses[i, 1, d]
                    tmp1[d] = a if a > b else b

                for j in range(C):
                    if learner_course_required_matching_njit(tmp1, courses[j]) >= threshold:
                        # tmp2 = max(tmp1, provided2)
                        for d in range(D):
                            a = tmp1[d]
                            b = courses[j, 1, d]
                            tmp2[d] = a if a > b else b

                        for k in range(C):
                            if learner_course_required_matching_njit(tmp2, courses[k]) >= threshold:
                                # tmp3 = max(tmp2, provided3)
                                for d in range(D):
                                    a = tmp2[d]
                                    b = courses[k, 1, d]
                                    tmp3[d] = a if a > b else b

                                new_jobs = _nb_applicable_jobs_numba(tmp3, jobs, threshold)
                                if new_jobs > best_jobs:
                                    best_jobs = new_jobs
                                    bi, bj, bk = i, j, k

        # applica eventuale miglior sequenza
        if bi != -1:
            out = learners[li].copy()
            # corso i
            for d in range(D):
                v = courses[bi, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso j
            for d in range(D):
                v = courses[bj, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso k
            for d in range(D):
                v = courses[bk, 1, d]
                out[d] = out[d] if out[d] > v else v

            final_learners[li] = out
            sequences[li, 0] = bi
            sequences[li, 1] = bj
            sequences[li, 2] = bk
        else:
            final_learners[li] = learners[li]

        max_jobs[li] = best_jobs
    print(f"Avg max jobs (3 courses): {sum(max_jobs) / len(max_jobs)}")

    return sum(max_jobs) / len(max_jobs)

# Esempio di utilizzo
'''max_jobs, sequences, final_learners = maximize_jobs_3courses_njit(
    learners=learners,
    courses=courses,
    jobs=jobs,
    threshold=0.8
)
print(f"Avg max jobs (3 courses): {sum(max_jobs) / len(max_jobs)}")'''


'max_jobs, sequences, final_learners = maximize_jobs_3courses_njit(\n    learners=learners,\n    courses=courses,\n    jobs=jobs,\n    threshold=0.8\n)\nprint(f"Avg max jobs (3 courses): {sum(max_jobs) / len(max_jobs)}")'

In [7]:
print(maximize_jobs_3courses_njit(learners, courses, jobs))

Avg max jobs (3 courses): <object type:float64>
23.057692307692307


In [15]:
total = []
for learner in learners:
    actual_jobs = dataset.get_nb_applicable_jobs(learner, 0.8)
    sequences = []
    max_job = actual_jobs

    for i, course1 in enumerate(courses):
        req1 = matchings.learner_course_required_matching(learner, course1)
        if req1 >= 0.8:
            learner_1course = np.maximum(learner, course1[1])
            new_max1 = dataset.get_nb_applicable_jobs(learner_1course, 0.8)

            for j, course2 in enumerate(courses):
                req2 = matchings.learner_course_required_matching(learner_1course, course2)
                if req2 >= 0.8:
                    learner_2course = np.maximum(learner_1course, course2[1])
                    new_max2 = dataset.get_nb_applicable_jobs(learner_2course, 0.8)

                    for k, course3 in enumerate(courses):
                        req3 = matchings.learner_course_required_matching(learner_2course, course3)
                        if req3 >= 0.8:
                            learner_3course = np.maximum(learner_2course, course3[1])
                            new_max3 = dataset.get_nb_applicable_jobs(learner_3course, 0.8)

                            if new_max3 > max_job:
                                max_job = new_max3
                                sequences = [i, j, k]

    # se ho trovato una sequenza, aggiorno lo stato del learner
    if sequences:
        learner = np.maximum(learner, courses[sequences[0]][1])
        learner = np.maximum(learner, courses[sequences[1]][1])
        learner = np.maximum(learner, courses[sequences[2]][1])

    total.append(max_job)
    print(f"For learner: {learner} we reached max_job: {max_job}, with sequence: {sequences}")
    print(sum(total) / len(total))

For learner: [2 2 2 2 2 2 2 2 2 2 2 2 2 0 1 0 2 0 2 2 0 2 0 0 2 1 2 2 1 0 0 0 0 1 0 0 2
 2 0 2 2 0 2 0 0 2] we reached max_job: 7, with sequence: [0, 24, 29]
7.0
For learner: [2 2 0 2 2 2 2 3 2 3 2 0 2 0 0 0 0 0 3 2 3 2 2 0 2 1 0 2 0 0 0 0 2 0 2 1 0
 2 0 0 2 0 2 0 0 3] we reached max_job: 3, with sequence: [22, 82, 29]
5.0
For learner: [2 0 0 3 3 2 2 2 2 2 0 2 2 0 0 3 3 0 2 2 0 2 0 0 2 1 0 0 0 0 0 3 0 0 0 2 3
 2 0 0 2 0 2 0 0 2] we reached max_job: 5, with sequence: [31, 12, 29]
5.0
For learner: [2 2 0 3 3 2 2 3 2 2 0 2 2 0 0 0 3 3 2 2 0 2 0 3 2 1 0 0 0 0 0 3 0 0 2 0 3
 3 0 0 2 0 2 0 0 2] we reached max_job: 8, with sequence: [12, 29, 41]
5.75
For learner: [2 1 0 2 2 2 2 2 2 2 2 0 2 0 0 0 0 0 2 2 0 2 2 0 2 1 0 2 0 0 0 0 0 0 0 1 0
 2 0 0 2 0 2 0 0 2] we reached max_job: 2, with sequence: [22, 82, 29]
5.0
For learner: [2 1 0 3 3 2 2 2 2 2 2 2 2 0 1 0 3 0 2 2 0 2 0 0 2 1 0 0 0 0 0 3 0 0 0 0 3
 2 0 2 2 0 2 0 0 2] we reached max_job: 5, with sequence: [0, 12, 29]
5.0
For learner: [2 2 0 3 3

KeyboardInterrupt: 

### 4 course sequence

In [8]:
import numpy as np
from numba import njit, prange

# =========================
#  Helpers Numba-friendly
# =========================

@njit(cache=True)
def vec_max_inplace(dst: np.ndarray, src: np.ndarray):
    """dst = maximum(dst, src) in-place (nopython-safe)."""
    for i in range(dst.size):
        a = dst[i]
        b = src[i]
        dst[i] = a if a > b else b

@njit(cache=True)
def matching_njit(level1: np.ndarray, level2: np.ndarray) -> float:
    """
    Versione nopython della tua 'matching' (media dei rapporti min/max su skill richieste).
    level1/level2: vettori (D,) con livelli 0–3.
    """
    acc = 0.0
    denom = 0
    for i in range(level1.size):
        req = level2[i]
        if req > 0.0:
            # min(level1, level2) / level2
            lv = level1[i]
            mn = lv if lv < req else req
            acc += (mn / req)
            denom += 1
    if denom == 0:
        return 0.0
    return acc / denom

@njit(cache=True)
def learner_course_required_matching_njit(learner: np.ndarray, course: np.ndarray) -> float:
    """
    Replica della tua 'learner_course_required_matching' in nopython.
    course: shape (2, D) -> [required, provided]
    """
    required_course = course[0]  # (D,)
    # se il corso non ha requisiti
    has_any = False
    for i in range(required_course.size):
        if required_course[i] != 0.0:
            has_any = True
            break
    if not has_any:
        return 1.0
    return matching_njit(learner, required_course)

# La tua funzione per contare i job applicabili è già njit.
# La ricopio qui per completezza (identica alla tua).
@njit(cache=True)
def _nb_applicable_jobs_numba(learner: np.ndarray, jobs: np.ndarray, threshold: float) -> int:
    """Compute the number of applicable jobs using Numba (speed-optimized)."""
    J, S = jobs.shape
    count = 0
    for j in range(J):  # loop over jobs
        denom = 0
        acc = 0.0
        for s in range(S):  # loop over skills
            req = jobs[j, s]
            if req > 0:  # required skill
                denom += 1
                lv = learner[s]
                acc += (lv if lv < req else req) / req
        if denom > 0:
            score = acc / denom  # average matching
            if score >= threshold:
                count += 1
    return count

# ===========================================
#  Kernel principale: miglior sequenza di 4
# ===========================================

@njit(parallel=True, cache=True)
def maximize_jobs_4courses_njit(
    learners: np.ndarray,        # (L, D)
    courses: np.ndarray,         # (C, 2, D) -> [required, provided]
    jobs: np.ndarray,            # (J, D)
    threshold: float = 0.8
):
    """
    Restituisce:
      - max_jobs: (L,) int64  -> #max jobs raggiunti
      - sequences: (L, 4) int64 -> indici corsi [i,j,k,l] o -1 se non usati
      - final_learners: (L, D) -> profilo learner dopo aver applicato la sequenza (se esiste)
    """
    L, D = learners.shape
    C = courses.shape[0]

    max_jobs = np.empty(L, dtype=np.int64)
    sequences = np.full((L, 4), -1, dtype=np.int64)
    final_learners = np.empty((L, D), dtype=learners.dtype)

    for li in prange(L):
        base = learners[li]  # (D,)
        best_jobs = _nb_applicable_jobs_numba(base, jobs, threshold)

        bi = -1
        bj = -1
        bk = -1
        bl = -1

        # buffer temporanei per evitare allocazioni ripetute
        tmp1 = np.empty(D, dtype=learners.dtype)
        tmp2 = np.empty(D, dtype=learners.dtype)
        tmp3 = np.empty(D, dtype=learners.dtype)
        tmp4 = np.empty(D, dtype=learners.dtype)

        for i in range(C):
            # check requisito corso1
            if learner_course_required_matching_njit(base, courses[i]) >= threshold:
                # tmp1 = max(base, provided1)
                for d in range(D):
                    a = base[d]
                    b = courses[i, 1, d]
                    tmp1[d] = a if a > b else b

                for j in range(C):
                    if learner_course_required_matching_njit(tmp1, courses[j]) >= threshold:
                        # tmp2 = max(tmp1, provided2)
                        for d in range(D):
                            a = tmp1[d]
                            b = courses[j, 1, d]
                            tmp2[d] = a if a > b else b

                        for k in range(C):
                            if learner_course_required_matching_njit(tmp2, courses[k]) >= threshold:
                                # tmp3 = max(tmp2, provided3)
                                for d in range(D):
                                    a = tmp2[d]
                                    b = courses[k, 1, d]
                                    tmp3[d] = a if a > b else b

                                for l in range(C):
                                    if learner_course_required_matching_njit(tmp3, courses[l]) >= threshold:
                                        # tmp4 = max(tmp3, provided4)
                                        for d in range(D):
                                            a = tmp3[d]
                                            b = courses[l, 1, d]
                                            tmp4[d] = a if a > b else b

                                        new_jobs = _nb_applicable_jobs_numba(tmp4, jobs, threshold)
                                        if new_jobs > best_jobs:
                                            best_jobs = new_jobs
                                            bi, bj, bk, bl = i, j, k, l

        # applica la miglior sequenza (se esiste)
        if bi != -1:
            out = learners[li].copy()
            # corso i
            for d in range(D):
                v = courses[bi, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso j
            for d in range(D):
                v = courses[bj, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso k
            for d in range(D):
                v = courses[bk, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso l
            for d in range(D):
                v = courses[bl, 1, d]
                out[d] = out[d] if out[d] > v else v

            final_learners[li] = out
            sequences[li, 0] = bi
            sequences[li, 1] = bj
            sequences[li, 2] = bk
            sequences[li, 3] = bl
        else:
            final_learners[li] = learners[li]

        max_jobs[li] = best_jobs
    print(f"Avg max jobs (4 courses): {sum(max_jobs) / len(max_jobs)}")
    return sum(max_jobs) / len(max_jobs)

# Esempio di utilizzo
'''max_jobs, sequences, final_learners = maximize_jobs_4courses_njit(
    learners=learners,
    courses=courses,
    jobs=jobs,
    threshold=0.8
)'''


'max_jobs, sequences, final_learners = maximize_jobs_4courses_njit(\n    learners=learners,\n    courses=courses,\n    jobs=jobs,\n    threshold=0.8\n)'

In [9]:
print(maximize_jobs_4courses_njit(learners, courses, jobs))

Avg max jobs (4 courses): <object type:float64>
0.0


In [7]:
total = []
for learner in learners:
    actual_jobs = dataset.get_nb_applicable_jobs(learner, 0.8)
    sequences = []
    max_job = actual_jobs

    for i, course1 in enumerate(courses):
        req1 = matchings.learner_course_required_matching(learner, course1)
        if req1 >= 0.8:
            learner_1course = np.maximum(learner, course1[1])
            new_max1 = dataset.get_nb_applicable_jobs(learner_1course, 0.8)

            for j, course2 in enumerate(courses):
                req2 = matchings.learner_course_required_matching(learner_1course, course2)
                if req2 >= 0.8:
                    learner_2course = np.maximum(learner_1course, course2[1])
                    new_max2 = dataset.get_nb_applicable_jobs(learner_2course, 0.8)

                    for k, course3 in enumerate(courses):
                        req3 = matchings.learner_course_required_matching(learner_2course, course3)
                        if req3 >= 0.8:
                            learner_3course = np.maximum(learner_2course, course3[1])
                            new_max3 = dataset.get_nb_applicable_jobs(learner_3course, 0.8)

                            # quarto corso
                            for l, course4 in enumerate(courses):
                                req4 = matchings.learner_course_required_matching(learner_3course, course4)
                                if req4 >= 0.8:
                                    learner_4course = np.maximum(learner_3course, course4[1])
                                    new_max4 = dataset.get_nb_applicable_jobs(learner_4course, 0.8)

                                    if new_max4 > max_job:
                                        max_job = new_max4
                                        sequences = [i, j, k, l]

    # se ho trovato una sequenza di 4 corsi che migliora, aggiorno lo stato del learner
    if sequences:
        learner = np.maximum(learner, courses[sequences[0]][1])
        learner = np.maximum(learner, courses[sequences[1]][1])
        learner = np.maximum(learner, courses[sequences[2]][1])
        learner = np.maximum(learner, courses[sequences[3]][1])

    total.append(max_job)
    print(f"For learner: {learner} we reached max_job: {max_job}, with sequence: {sequences}")

print(sum(total) / len(total))


For learner: [2 2 2 2 2 2 2 2 2 2 0 2 2 0 1 0 2 0 2 2 2 2 2 0 2 1 2 2 1 0 0 1 0 1 0 2 2
 2 0 0 2 0 2 0 0 2] we reached max_job: 12, with sequence: [13, 29, 31, 77]


KeyboardInterrupt: 

### 5 course reccomendation

In [ ]:
import numpy as np
from numba import njit, prange

@njit(parallel=True, cache=True)
def maximize_jobs_5courses_njit(
    learners: np.ndarray,        # (L, D)
    courses: np.ndarray,         # (C, 2, D) -> [required, provided]
    jobs: np.ndarray,            # (J, D)
    threshold: float = 0.8
):
    """
    Cerca la miglior sequenza di 5 corsi.
    Output:
      - max_jobs: (L,) int64
      - sequences: (L, 5) int64  -> indici [i,j,k,l,m] o -1 se nessuna sequenza migliora
      - final_learners: (L, D)
    """
    L, D = learners.shape
    C = courses.shape[0]

    max_jobs = np.empty(L, dtype=np.int64)
    sequences = np.full((L, 5), -1, dtype=np.int64)
    final_learners = np.empty((L, D), dtype=learners.dtype)

    for li in prange(L):
        base = learners[li]
        best_jobs = _nb_applicable_jobs_numba(base, jobs, threshold)
        bi = bj = bk = bl = bm = -1

        # buffer per evitare allocazioni ripetute
        tmp1 = np.empty(D, dtype=learners.dtype)
        tmp2 = np.empty(D, dtype=learners.dtype)
        tmp3 = np.empty(D, dtype=learners.dtype)
        tmp4 = np.empty(D, dtype=learners.dtype)
        tmp5 = np.empty(D, dtype=learners.dtype)

        for i in range(C):
            if learner_course_required_matching_njit(base, courses[i]) >= threshold:
                # tmp1 = max(base, provided1)
                for d in range(D):
                    a = base[d]
                    b = courses[i, 1, d]
                    tmp1[d] = a if a > b else b

                for j in range(C):
                    if learner_course_required_matching_njit(tmp1, courses[j]) >= threshold:
                        # tmp2 = max(tmp1, provided2)
                        for d in range(D):
                            a = tmp1[d]
                            b = courses[j, 1, d]
                            tmp2[d] = a if a > b else b

                        for k in range(C):
                            if learner_course_required_matching_njit(tmp2, courses[k]) >= threshold:
                                # tmp3 = max(tmp2, provided3)
                                for d in range(D):
                                    a = tmp2[d]
                                    b = courses[k, 1, d]
                                    tmp3[d] = a if a > b else b

                                for l in range(C):
                                    if learner_course_required_matching_njit(tmp3, courses[l]) >= threshold:
                                        # tmp4 = max(tmp3, provided4)
                                        for d in range(D):
                                            a = tmp3[d]
                                            b = courses[l, 1, d]
                                            tmp4[d] = a if a > b else b

                                        for m in range(C):
                                            if learner_course_required_matching_njit(tmp4, courses[m]) >= threshold:
                                                # tmp5 = max(tmp4, provided5)
                                                for d in range(D):
                                                    a = tmp4[d]
                                                    b = courses[m, 1, d]
                                                    tmp5[d] = a if a > b else b

                                                new_jobs = _nb_applicable_jobs_numba(tmp5, jobs, threshold)
                                                if new_jobs > best_jobs:
                                                    best_jobs = new_jobs
                                                    bi, bj, bk, bl, bm = i, j, k, l, m

        # applica eventuale miglior sequenza
        if bi != -1:
            out = learners[li].copy()
            # corso i
            for d in range(D):
                v = courses[bi, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso j
            for d in range(D):
                v = courses[bj, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso k
            for d in range(D):
                v = courses[bk, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso l
            for d in range(D):
                v = courses[bl, 1, d]
                out[d] = out[d] if out[d] > v else v
            # corso m
            for d in range(D):
                v = courses[bm, 1, d]
                out[d] = out[d] if out[d] > v else v

            final_learners[li] = out
            sequences[li, 0] = bi
            sequences[li, 1] = bj
            sequences[li, 2] = bk
            sequences[li, 3] = bl
            sequences[li, 4] = bm
        else:
            final_learners[li] = learners[li]

        max_jobs[li] = best_jobs

    return max_jobs, sequences, final_learners
# Esempio di utilizzo
max_jobs, sequences, final_learners = maximize_jobs_5courses_njit(
    learners=learners,
    courses=courses,
    jobs=jobs,
    threshold=0.8
)
print(f"Avg max jobs (5 courses): {sum(max_jobs) / len(max_jobs)}")

In [5]:
print(maximize_jobs_2courses(learners, courses, dataset))

2.480769230769231
2.480769230769231


In [7]:
seed = []

seed.append(maximize_jobs_2courses(learners, courses, dataset))
seed.append(maximize_jobs_3courses_njit(learners, courses, jobs, threshold=0.8))
seed.append(maximize_jobs_4courses_njit(learners, courses, jobs, threshold=0.8))

print("Seed results:", seed)

2.5576923076923075
Avg max jobs (3 courses): <object type:float64>
Avg max jobs (4 courses): <object type:float64>
Seed results: [2.5576923076923075, 4.615384615384615, 6.0576923076923075]


In [7]:
seedLWM = []

seedLWM.append(maximize_jobs_2courses(learners, courses, dataset))
seedLWM.append(maximize_jobs_3courses_njit(learners, courses, jobs, threshold=0.8))
seedLWM.append(maximize_jobs_4courses_njit(learners, courses, jobs, threshold=0.8))

print("SeedLWM results:", seedLWM)

2.576923076923077
Avg max jobs (3 courses): <object type:float64>
Avg max jobs (4 courses): <object type:float64>
SeedLWM results: [2.576923076923077, 4.615384615384615, 6.0576923076923075]


In [1]:
seed = [2.5, 5.615384615384615, 9.961538461538462]
seedLWM = [2.480769230769231, 5.615384615384615, 9.961538461538462]

seed43 = [2.1346153846153846, 4.153846153846154, 6.038461538461538]
seedLWM43 = [2.173076923076923, 4.153846153846154, 6.038461538461538]

seed44 = [2.980769230769231, 6.288461538461538, 11.673076923076923]
seedLWM44 = [2.8076923076923075, 6.288461538461538, 11.673076923076923]

seed45 = [1.4230769230769231, 4.980769230769231, 9.98076923076923]
seedLWM45 = [1.4230769230769231, 4.980769230769231, 9.98076923076923]

seed46 = [2.3076923076923075, 3.6153846153846154, 5.153846153846154]
seedLWM46 = [2.1346153846153846, 3.6153846153846154, 5.153846153846154]

seed47 = [0.5576923076923077, 1.3846153846153846, 2.269230769230769]
seedLWM47 = [0.5576923076923077, 1.3846153846153846, 2.269230769230769]

seed48 = [1.7115384615384615, 3.7884615384615383, 6.75]
seedLWM48 = [1.6153846153846154, 3.7884615384615383, 6.75]

seed49 = [2.326923076923077, 5.384615384615385, 8.134615384615385]
seedLWM49 = [2.3076923076923075, 5.384615384615385, 8.134615384615385]

seed50 = [2.5576923076923075, 4.615384615384615, 6.0576923076923075]
seedLWM50 = [2.576923076923077, 4.615384615384615, 6.0576923076923075]

In [2]:
# Calculate standard deviation of seed results between LWM and non-LWM considering all seeds in all the previous lists pairwisely 
import numpy as np
all_seeds = [seed, seedLWM, seed43, seedLWM43, seed44, seedLWM44, seed45, seedLWM45, seed46, seedLWM46, seed47, seedLWM47, seed48, seedLWM48, seed49, seedLWM49, seed50, seedLWM50]
std_devs = []
for i in range(0, len(all_seeds), 2):
    std_dev = np.std(np.array(all_seeds[i]) - np.array(all_seeds[i+1]))
    std_devs.append(std_dev)
#Performe mean of the standard deviations elementwise across all the calculated standard deviations
mean_std_dev = np.mean(std_devs)
print("Mean standard deviation between seed results and seedLWM results across all seeds:", mean_std_dev)

# Perform 3 mean of the standard deviations elementwise considering indices 0,3,6,... and 1,4,7,... 2,5,8 separately
std_dev_0 = np.std([all_seeds[i][0] - all_seeds[i+1][0] for i in range(0, len(all_seeds), 2)])
std_dev_1 = np.std([all_seeds[i][1] - all_seeds[i+1][1] for i in range(0, len(all_seeds), 2)])
std_dev_2 = np.std([all_seeds[i][2] - all_seeds[i+1][2] for i in range(0, len(all_seeds), 2)])
print("Standard deviation for k=2 across all seeds:", std_dev_0)
print("Standard deviation for k=3 across all seeds:", std_dev_1)
print("Standard deviation for k=4 across all seeds:", std_dev_2)

Mean standard deviation between seed results and seedLWM results across all seeds: 0.028203689278095943
Standard deviation for k=2 across all seeds: 0.07590734900996417
Standard deviation for k=3 across all seeds: 0.0
Standard deviation for k=4 across all seeds: 0.0
